# kaiming-uniform-sf-init — worked example 2: SF init — same generator seed gives same weights

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `kaiming-uniform-sf-init`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The `generator` parameter to `t.rand` controls the random state for weight sampling. Re-seeding the same `Generator` object and calling the initializer again must produce bit-for-bit identical tensors. This reproducibility is essential for debugging and ablation experiments.

## Worked solution

Step 1: Create a `t.Generator()` and call `g.manual_seed(99)` before the first call to `kaiming_uniform_sf`.

Step 2: Store the resulting weight tensor `w1`.

Step 3: Reset the same generator with `g.manual_seed(99)` and call the function again to produce `w2`.

Step 4: Confirm `t.equal(w1, w2)` is True — same seed must produce identical outputs.

Step 5: Seed to a different value and confirm the new tensor differs from `w1`.

In [ ]:
import torch as t

def kaiming_uniform_sf(in_features: int, out_features: int, generator: t.Generator) -> t.Tensor:
    sf = in_features ** -0.5
    return (t.rand(in_features, out_features, generator=generator) * 2 - 1) * sf

g = t.Generator()

# First sample
g.manual_seed(99)
w1 = kaiming_uniform_sf(16, 8, g)

# Same seed -> identical output
g.manual_seed(99)
w2 = kaiming_uniform_sf(16, 8, g)

print('w1 == w2 (same seed):', t.equal(w1, w2))  # True

# Different seed -> different output
g.manual_seed(100)
w3 = kaiming_uniform_sf(16, 8, g)
print('w1 == w3 (diff seed):', t.equal(w1, w3))  # False

# Shape check
print('shape:', w1.shape)  # (16, 8)